In [ ]:
import torch
from torch import nn
class residual(nn.Module):
    def __init__(self,in_channels,out_channels,use_1x1conv=False,stride=1):
        super().__init__()
        self.main_branch = nn.Sequential(
            nn.Conv2d(in_channels=in_channels,out_channels=out_channels,stride=stride,kernel_size=3,padding=1),
            nn.BatchNorm2d(num_features=out_channels),
            nn.ReLU(),
            nn.Conv2d(in_channels=out_channels,out_channels=out_channels,stride=1,kernel_size=3,padding=1),
            nn.BatchNorm2d(num_features=out_channels)
        )
        if use_1x1conv:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels=in_channels,out_channels=out_channels,stride=stride,kernel_size=1)
            )
        else:
            self.shortcut = nn.Identity()
        self.relu = nn.ReLU()
    def forward(self,x):
        return self.relu(self.main_branch(x)+self.shortcut(x))
    
x = torch.randn(2, 16, 32, 32)

# 不改变形状
block1 = residual(16, 16)
print(block1(x).shape)

# 通道翻倍、空间尺寸减半
block2 = residual(
    16, 32,
    use_1x1conv=True,
    stride=2
)
print(block2(x).shape)


torch.Size([2, 16, 32, 32])
torch.Size([2, 32, 16, 16])


In [9]:
def resnet_stage(in_channels,out_channels,num_residuals,first_stage=False):
    layer = []
    if first_stage:
        for i in range(num_residuals):
            layer.append(residual(in_channels,out_channels,stride=1))
    else:
        layer.append(residual(in_channels,out_channels,use_1x1conv=True,stride=2))
        for i in range(num_residuals-1):
            layer.append(residual(out_channels,out_channels,stride=1))
    return nn.Sequential(*layer)

x = torch.randn(2, 64, 24, 24)

stage1 = resnet_stage(64, 64, 2, first_stage=True)
stage2 = resnet_stage(64, 128, 2)

y = stage1(x)
z = stage2(y)

print(y.shape)
print(z.shape) 

torch.Size([2, 64, 24, 24])
torch.Size([2, 128, 12, 12])


In [12]:
x = torch.randn(2, 1, 96, 96)


#Conv 7×7，64 通道，stride=2，padding=3；BN；ReLU；MaxPool 3×3，stride=2，padding=1

stem = nn.Sequential(
    nn.Conv2d(in_channels=1,out_channels=64,kernel_size=7,stride=2,padding=3),
    nn.BatchNorm2d(num_features=64),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3,stride=2,padding=1)
)

classify = nn.Sequential(
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(512,10)
)


net = nn.Sequential(
    stem,
    resnet_stage(64,64,2,first_stage=True),
    resnet_stage(64,128,2),
    resnet_stage(128,256,2),
    resnet_stage(256,512,2),
    classify
    
)
with torch.no_grad():
    for layer in net:
        x = layer(x)
        print(type(layer).__name__, x.shape)


Sequential torch.Size([2, 64, 24, 24])
Sequential torch.Size([2, 64, 24, 24])
Sequential torch.Size([2, 128, 12, 12])
Sequential torch.Size([2, 256, 6, 6])
Sequential torch.Size([2, 512, 3, 3])
Sequential torch.Size([2, 10])
